# Multimodal text–image embeddings with vision language models

**Run in Google Colab only** — this notebook is not intended for local Jupyter or `uv sync`.

**What you will learn**

- Load a **vision language model (VLM)** embedding model ([Qwen3-VL-Embedding-2B](https://huggingface.co/Qwen/Qwen3-VL-Embedding-2B)) with [Sentence Transformers](https://www.sbert.net/)
- Encode **text** and **image** inputs into one shared vector space
- Measure **cross-modal similarity** (text query vs image documents) and use `encode_query` / `encode_document` for retrieval-style APIs

**Guide:** [Multimodal Embedding & Reranker Models with Sentence Transformers](https://huggingface.co/blog/multimodal-sentence-transformers) (Hugging Face blog).

**How to open (only supported path)**

1. [Google Colab](https://colab.research.google.com/) → **File → Open notebook → GitHub**
2. Repo: `ysskrishna/awesome-llm-experiments` → `experiments/multimodal-text-image-vl-embeddings/notebook.ipynb`
3. **Runtime → Change runtime type → T4 GPU** (required; model needs ~8 GB VRAM)
4. Run all cells **top to bottom**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ysskrishna/awesome-llm-experiments/blob/main/experiments/multimodal-text-image-vl-embeddings/notebook.ipynb)

**First run:** [`Qwen/Qwen3-VL-Embedding-2B`](https://huggingface.co/Qwen/Qwen3-VL-Embedding-2B) downloads from Hugging Face on first load (multi‑GB; exact size depends on format). Cached under `~/.cache/huggingface/hub/` in the Colab VM. Expect several minutes for install + download + first encode on a T4.

## Concepts (quick links)

| Idea | Link |
|------|------|
| Multimodal / cross-modal embeddings | [HF blog: multimodal sentence transformers](https://huggingface.co/blog/multimodal-sentence-transformers) |
| Qwen3-VL embedding model | [Qwen3-VL-Embedding-2B](https://huggingface.co/Qwen/Qwen3-VL-Embedding-2B) |
| Sentence Transformers API | [Documentation](https://www.sbert.net/) |
| Modality gap (lower cross-modal scores) | See demo B below |

**Flow:** install deps → verify GPU → define demo URLs → load VLM → encode images/text → similarity matrix → known-answer rank checks → `encode_query` / `encode_document` retrieval demo.

## 1. Colab environment — install dependencies and verify GPU

Installs `sentence-transformers` with the **image** extra (v5.4+ for multimodal encode). Fails fast if you are not in Colab or if no GPU is available.

In [ ]:
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not IN_COLAB:
    raise RuntimeError(
        "This notebook is Colab-only. Open it from GitHub in Google Colab "
        "(File → Open notebook → GitHub) and enable a GPU runtime."
    )

In [ ]:
%pip install -q -U "sentence-transformers[image]>=5.4"

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU not available. In Colab: Runtime → Change runtime type → "
        "Hardware accelerator → T4 GPU, then restart and run all cells."
    )

print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Define helpers and demo constants

Defines `MODEL_NAME`, demo image URLs (from the HF blog), and small utilities for printing similarity matrices and rank checks. **Does not load the model yet.**

In [ ]:
MODEL_NAME = "Qwen/Qwen3-VL-Embedding-2B"

CAR_IMAGE = (
    "https://huggingface.co/datasets/huggingface/documentation-images/"
    "resolve/main/transformers/tasks/car.jpg"
)
BEE_IMAGE = (
    "https://huggingface.co/datasets/huggingface/documentation-images/"
    "resolve/main/bee.jpg"
)
DEMO_IMAGES = [CAR_IMAGE, BEE_IMAGE]
IMAGE_LABELS = ["car", "bee"]


def print_similarity_matrix(similarities, row_labels, col_labels):
    """Print text×image similarity scores with row/column labels."""
    sim = similarities.cpu()
    header = " " * 24 + "  ".join(f"{c:>8}" for c in col_labels)
    print(header)
    for i, row_name in enumerate(row_labels):
        scores = "  ".join(f"{sim[i, j].item():8.4f}" for j in range(sim.shape[1]))
        print(f"{row_name:24} {scores}")


def best_image_index(similarities, text_row: int, num_images: int) -> int:
    """Index of the image column with highest similarity for one text row."""
    row = similarities[text_row, :num_images]
    return int(row.argmax().item())


def assert_text_prefers_image(similarities, text_row, expected_image_idx, caption):
    best = best_image_index(similarities, text_row, similarities.shape[1])
    assert best == expected_image_idx, (
        f"{caption}: expected image index {expected_image_idx}, got {best}"
    )

## 3. Load the vision language embedding model

Downloads and loads **Qwen3-VL-Embedding-2B** on GPU. This cell is the slow step on first run.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(MODEL_NAME)
assert model.supports("image"), "Expected image modality support"
print(f"Modalities: {model.modalities}")

## 4. Encode images — expect shape (2, embedding_dim)

Encodes two demo images from URLs. Embedding dimension is model-specific (2048 for this Qwen3-VL checkpoint per the [blog](https://huggingface.co/blog/multimodal-sentence-transformers)).

In [ ]:
img_embeddings = model.encode(DEMO_IMAGES)
print(f"Image embeddings shape: {img_embeddings.shape}")
assert img_embeddings.shape[0] == len(DEMO_IMAGES)
assert img_embeddings.shape[1] > 0

## 5. Cross-modal similarity — text queries vs image documents

Same car/bee example as the HF blog: two images, four text captions (match + hard negative per image). We assert **ranking** (which image wins), not absolute score thresholds.

**Modality gap:** cross-modal similarity scores are often lower than within-modal scores (e.g. text–text), but relative order still supports retrieval.

In [ ]:
texts = [
    "A green car parked in front of a yellow building",
    "A red car driving on a highway",
    "A bee on a pink flower",
    "A wasp on a wooden table",
]
text_labels = ["car_match", "car_negative", "bee_match", "bee_negative"]

img_embeddings = model.encode(DEMO_IMAGES)
text_embeddings = model.encode(texts)
similarities = model.similarity(text_embeddings, img_embeddings)

print_similarity_matrix(similarities, text_labels, IMAGE_LABELS)

assert_text_prefers_image(similarities, 0, 0, "Green car caption")
assert_text_prefers_image(similarities, 2, 1, "Bee on flower caption")
print("Rank checks passed: car caption → car image; bee caption → bee image.")

## 6. Retrieval-style API — `encode_query` and `encode_document`

Many retrieval models apply different prompts for queries vs documents. These methods wrap `encode()` with the model’s configured prompts when available.

In [ ]:
queries = [
    "Find me a photo of a vehicle parked near a building",
    "Show me an image of a pollinating insect",
]

query_embeddings = model.encode_query(queries)
doc_embeddings = model.encode_document(DEMO_IMAGES)
retrieval_sims = model.similarity(query_embeddings, doc_embeddings)

print_similarity_matrix(
    retrieval_sims,
    ["vehicle_query", "insect_query"],
    IMAGE_LABELS,
)

assert best_image_index(retrieval_sims, 0, 2) == 0, "Vehicle query should rank car image first"
assert best_image_index(retrieval_sims, 1, 2) == 1, "Insect query should rank bee image first"
print("Retrieval rank checks passed.")

## Wrap-up

You loaded a **multimodal VLM embedder**, encoded **text** and **images** into one space, and verified **cross-modal** ranking on a tiny demo corpus.

**If you hit OOM on Colab:** reload with lower memory, e.g. `SentenceTransformer(MODEL_NAME, model_kwargs={"torch_dtype": "bfloat16"})` (see the [blog’s processor/model kwargs section](https://huggingface.co/blog/multimodal-sentence-transformers#processor-and-model-kwargs)).

**Next steps (not covered here)**

- [Multimodal rerankers](https://huggingface.co/blog/multimodal-sentence-transformers#multimodal-reranker-models) (`CrossEncoder`) for higher-quality rescoring
- [Training multimodal embedders](https://huggingface.co/blog/train-multimodal-sentence-transformers)
- Index image/document embeddings in a vector DB for multimodal RAG